# 21 - Introduccion a ciencia de datos en R: CRISP-DM y prediccion con Iris

## Proposito de la sesion

En esta sesion vas a:

1. Entender CRISP-DM como proceso de trabajo.
2. Aclarar que significa predecir (y que no significa).
3. Construir un predictor simple con `if/else`.
4. Entrenar un modelo k-NN en Iris con flujo train/test.
5. Visualizar regiones de decision.
6. Entender la idea de densidad de datos de forma intuitiva.


## CRISP-DM en lenguaje practico

CRISP-DM organiza un proyecto de ciencia de datos en 6 fases:

1. Entendimiento del negocio.
2. Entendimiento de los datos.
3. Preparacion de datos.
4. Modelado.
5. Evaluacion.
6. Despliegue.

La clave: es iterativo. Se vuelve a fases previas cuando hay hallazgos nuevos.


In [ ]:
# Ejemplo minimo: representar CRISP-DM como tabla de trabajo.
crisp_dm <- data.frame(
  fase = c('Negocio', 'Datos', 'Preparacion', 'Modelado', 'Evaluacion', 'Despliegue'),
  pregunta_clave = c(
    'Que decision queremos mejorar?',
    'Que calidad tienen los datos?',
    'Que variables y limpieza se requieren?',
    'Que algoritmo usar y por que?',
    'Sirve en datos no vistos?',
    'Como se usa y monitorea en operacion?'
  )
)

crisp_dm


## Que significa predecir

Predecir es estimar una salida para datos nuevos usando patrones aprendidos:

- Clasificacion: salida categorica (ej. especie de flor).
- Regresion: salida numerica (ej. precio).

Predecir no es adivinar con certeza absoluta: siempre hay incertidumbre.


In [ ]:
# Ejemplo rapido de clasificacion basada en regla.
# Si petalo es corto, clasificamos como setosa.
clasificar_simple <- function(petal_length) {
  if (petal_length < 2.45) 'setosa' else 'no_setosa'
}

clasificar_simple(1.9)
clasificar_simple(4.8)


## Cargar Iris y revisar datos

Trabajaremos con `Petal.Length` y `Petal.Width` porque separan bien las clases.


In [ ]:
# Carga de datos Iris desde R base.
X <- iris[, c('Petal.Length', 'Petal.Width')]
y <- iris$Species

# Exploracion minima.
str(iris)
summary(X)
table(y)


## Visualizacion inicial

Antes de modelar, ver la geometria del problema ayuda a intuir separaciones y solapamientos.


In [ ]:
# Grafica de dispersion por especie.
cols <- c(setosa = 'steelblue', versicolor = 'darkorange', virginica = 'darkgreen')
plot(
  X$Petal.Length,
  X$Petal.Width,
  col = cols[y],
  pch = 19,
  xlab = 'Petal.Length',
  ylab = 'Petal.Width',
  main = 'Iris en dos variables'
)
legend('topleft', legend = names(cols), col = cols, pch = 19)
grid()


## Baseline interpretable: regla `if/else`

Un baseline simple y explicable:

1. Si `Petal.Length < 2.45` -> setosa.
2. Si no, y `Petal.Width < 1.75` -> versicolor.
3. En otro caso -> virginica.


In [ ]:
# Regla manual interpretable.
predict_rule <- function(pl, pw) {
  if (pl < 2.45) {
    return('setosa')
  } else if (pw < 1.75) {
    return('versicolor')
  } else {
    return('virginica')
  }
}

# Prediccion de la regla sobre todo el dataset.
pred_rule <- mapply(predict_rule, X$Petal.Length, X$Petal.Width)

# Exactitud global del baseline.
acc_rule <- mean(pred_rule == as.character(y))
acc_rule


## Evaluacion: accuracy y matriz de confusion

- Accuracy resume aciertos globales.
- Matriz de confusion muestra en que clases exactas se equivoca el modelo.


In [ ]:
# Matriz de confusion de la regla manual.
cm_rule <- table(real = y, pred = pred_rule)
cm_rule

# Accuracy desde la matriz.
acc_rule_from_cm <- sum(diag(cm_rule)) / sum(cm_rule)
acc_rule_from_cm


## Flujo train/test con k-NN

Ahora usamos un modelo basado en vecinos para comparar contra la regla manual.
Buenas practicas:

1. Separar train/test.
2. Escalar usando solo train.
3. Evaluar en test no visto.


In [ ]:
# Carga k-NN desde paquete recomendado.
library(class)

# Split reproducible.
set.seed(2026)
idx <- sample(seq_len(nrow(X)), size = floor(0.75 * nrow(X)))
X_train <- X[idx, ]
X_test <- X[-idx, ]
y_train <- y[idx]
y_test <- y[-idx]

# Escalado sin fuga de informacion.
mu <- colMeans(X_train)
sd_ <- apply(X_train, 2, sd)
scale_with <- function(M, mu, sd_) {
  sweep(sweep(as.matrix(M), 2, mu, '-'), 2, sd_, '/')
}

X_train_sc <- scale_with(X_train, mu, sd_)
X_test_sc <- scale_with(X_test, mu, sd_)

# Modelo k-NN.
k <- 11
pred_knn <- knn(train = X_train_sc, test = X_test_sc, cl = y_train, k = k)

# Metricas basicas.
cm_knn <- table(real = y_test, pred = pred_knn)
acc_knn <- mean(pred_knn == y_test)
cm_knn
acc_knn


## Comparacion baseline vs k-NN

Comparar modelos simples vs modelos de vecindad ayuda a decidir con evidencia.


In [ ]:
# Prediccion de la regla manual en test.
pred_rule_test <- mapply(predict_rule, X_test$Petal.Length, X_test$Petal.Width)
acc_rule_test <- mean(pred_rule_test == as.character(y_test))

# Reporte comparativo.
comparacion <- data.frame(
  modelo = c('Regla if/else', 'k-NN (k=11)'),
  accuracy_test = c(acc_rule_test, acc_knn)
)
comparacion


## Regiones de decision (intuicion visual)

Muchos algoritmos de clasificacion dividen el espacio de variables en regiones.
Aqui mostramos regiones predichas por k-NN.


In [ ]:
# Construye una malla de puntos para visualizar regiones.
x_seq <- seq(min(X$Petal.Length) - 0.5, max(X$Petal.Length) + 0.5, length.out = 180)
y_seq <- seq(min(X$Petal.Width) - 0.3, max(X$Petal.Width) + 0.3, length.out = 180)
grid <- expand.grid(Petal.Length = x_seq, Petal.Width = y_seq)

# Escala la malla con parametros de train.
grid_sc <- scale_with(grid, mu, sd_)

# Predice clase en cada punto de la malla.
pred_grid <- knn(train = X_train_sc, test = grid_sc, cl = y_train, k = k)
z <- matrix(as.integer(pred_grid), nrow = length(x_seq), ncol = length(y_seq))

# Dibuja regiones + datos reales.
image(
  x_seq,
  y_seq,
  z,
  col = adjustcolor(c('steelblue', 'darkorange', 'darkgreen'), alpha.f = 0.25),
  xlab = 'Petal.Length',
  ylab = 'Petal.Width',
  main = 'Regiones de decision k-NN'
)
points(X$Petal.Length, X$Petal.Width, col = cols[y], pch = 19)
legend('topleft', legend = names(cols), col = cols, pch = 19)


## Densidad de datos (intuicion)

La densidad indica donde los valores son mas frecuentes.
Cuando dos densidades se solapan, la clasificacion suele ser mas incierta.


In [ ]:
# Densidad de Petal.Length por especie.
plot(
  density(iris$Petal.Length[y == 'setosa']),
  col = 'steelblue',
  lwd = 2,
  main = 'Densidad por especie (Petal.Length)',
  xlab = 'Petal.Length'
)
lines(density(iris$Petal.Length[y == 'versicolor']), col = 'darkorange', lwd = 2)
lines(density(iris$Petal.Length[y == 'virginica']), col = 'darkgreen', lwd = 2)
legend(
  'topright',
  legend = c('setosa', 'versicolor', 'virginica'),
  col = c('steelblue', 'darkorange', 'darkgreen'),
  lty = 1,
  lwd = 2
)
grid()


## Extra de probabilidad y estadistica

Estimaremos probabilidades locales de clase usando vecinos cercanos.
Idea: la proporcion de clases entre vecinos puede leerse como evidencia probabilistica local.


In [ ]:
# Funcion para probabilidades locales por vecindad.
local_probs <- function(punto, X_train_sc, y_train, k = 15) {
  # Distancias euclidianas al punto.
  d <- sqrt(rowSums((X_train_sc - matrix(punto, nrow(X_train_sc), ncol(X_train_sc), byrow = TRUE))^2))

  # Vecinos mas cercanos.
  idx_k <- order(d)[1:k]
  y_k <- y_train[idx_k]

  # Proporciones por clase.
  prop.table(table(y_k))
}

# Puntos de consulta (en escala original).
consultas <- data.frame(
  Petal.Length = c(1.5, 4.8, 5.8),
  Petal.Width = c(0.2, 1.5, 2.0)
)

# Escala consultas y estima probabilidades locales.
consultas_sc <- scale_with(consultas, mu, sd_)
probs_1 <- local_probs(consultas_sc[1, ], X_train_sc, y_train, k = 15)
probs_2 <- local_probs(consultas_sc[2, ], X_train_sc, y_train, k = 15)
probs_3 <- local_probs(consultas_sc[3, ], X_train_sc, y_train, k = 15)

probs_1
probs_2
probs_3


## Analogias rapidas R y Python

- `if/else` funciona de forma muy parecida en ambos lenguajes.
- `class::knn` en R cumple una funcion similar a `KNeighborsClassifier` en Python.
- El principio estadistico de evaluar en test es igual en ambos ecosistemas.


In [ ]:
# Ejemplo rapido de pipeline conceptual equivalente en R.
# 1) Split
# 2) Escalado con train
# 3) Entrenamiento
# 4) Evaluacion

cat('Pipeline en R: split -> scale(train) -> knn -> evaluate
')


## Errores comunes en esta sesion

1. Hacer escalado antes del split (data leakage).
2. Elegir modelo solo por accuracy sin revisar confusion por clase.
3. Interpretar densidad como causalidad.
4. No fijar semilla al comparar metodos.


In [ ]:
# Demostracion de fuga de informacion (solo para ilustrar).
# Correcto: usar mu/sd de train.
# Incorrecto: usar mu/sd de todo el dataset antes del split.

mu_all <- colMeans(X)
sd_all <- apply(X, 2, sd)
X_train_leak <- scale_with(X_train, mu_all, sd_all)
X_test_leak <- scale_with(X_test, mu_all, sd_all)

pred_leak <- knn(train = X_train_leak, test = X_test_leak, cl = y_train, k = k)
acc_leak <- mean(pred_leak == y_test)

cat('Accuracy con fuga (solo referencia):', round(acc_leak, 4), '
')
cat('Accuracy sin fuga:', round(acc_knn, 4), '
')


## Ejercicios para pensar (no copiar codigo)

1. Redacta un caso real y recorre sus 6 fases CRISP-DM con decisiones concretas.
2. Ajusta umbrales de la regla manual y explica como cambia el tradeoff interpretabilidad vs accuracy.
3. Compara `k = 3`, `k = 11` y `k = 25`. ?Que esperas en sesgo/varianza?
4. Elige dos puntos cerca de fronteras y explica por que su prediccion es incierta.
5. Dise?a una metrica adicional a accuracy que usar?as en este problema y justifica.
6. Explica a una persona no tecnica por que evaluar en test evita conclusiones enga?osas.


In [ ]:
# Espacio para resolver ejercicios en R.
# Escribe aqui tus respuestas, pruebas y conclusiones.
